# 12 - Multi-Modal RAG


## What this notebook is for

This notebook is written as **notes + code**. Read the Markdown cells first, then run the code cells.

The goal is to understand the concept, see the architecture, implement a small example, and then evaluate/improve it.

## Learning objectives

- Understand text-only vs multimodal RAG.
- Retrieve evidence from text and visual content.
- Use image inputs with a vision-capable model.
- Separate image retrieval from image understanding.

## Core AI concept

Multimodal RAG retrieves information that may include text, images, tables, charts, or page images. A practical architecture often extracts text plus visual descriptions/embeddings, retrieves candidates, then sends the most relevant text and images to a multimodal model.

## Architecture / mental model

```text
PDF / images / tables
      ↓
text extraction + image extraction
      ↓
text/image descriptions + embeddings
      ↓
hybrid retrieval
      ↓
text + image evidence
      ↓
vision-capable LLM
      ↓
answer + citations
```

## Connection to the supplied learning material

Your original RAG notebook is text-focused. This notebook extends the same retrieve→context→generate pattern to visual evidence.

### Source foundation
This notebook is an expanded learning version of the uploaded notebooks. The source material for this track includes:
- `chunking.ipynb`
- `embeddings.ipynb`
- `faiss.ipynb`
- `chroma_db.ipynb`
- `searching.ipynb`
- `rag.ipynb`

The original examples and terminology are retained where useful, then organized into a cleaner learning sequence.

## Setup

The code below uses environment variables for secrets and model names so the notebook is easier to maintain when providers or model versions change.

In [ ]:
from pathlib import Path
import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.6-luna")
OPENAI_EMBED_MODEL = os.getenv("OPENAI_EMBED_MODEL", "text-embedding-3-small")

print("Model:", OPENAI_MODEL)
print("Embedding model:", OPENAI_EMBED_MODEL)

In [ ]:
%pip install -qU openai pillow chromadb

### Create a simple image

In [ ]:
from PIL import Image, ImageDraw

img = Image.new("RGB", (600, 300), "white")
draw = ImageDraw.Draw(img)
draw.rectangle((50, 50, 550, 250), outline="black", width=3)
draw.text((90, 130), "RAG can retrieve visual evidence", fill="black")
img.save("sample_diagram.png")

display(img)

### Vision input with the Responses API

In [ ]:
import base64
from openai import OpenAI

client = OpenAI()

with open("sample_diagram.png", "rb") as f:
    b64 = base64.b64encode(f.read()).decode("utf-8")

response = client.responses.create(
    model=os.getenv("VISION_MODEL", OPENAI_MODEL),
    input=[{
        "role": "user",
        "content": [
            {"type": "input_text", "text": "Describe what this image contains."},
            {"type": "input_image", "image_url": f"data:image/png;base64,{b64}"},
        ],
    }]
)

print(response.output_text)

### Multimodal retrieval pattern

In [ ]:
multimodal_record = {
    "id": "img-001",
    "text": "Diagram showing a RAG pipeline",
    "image_path": "sample_diagram.png",
    "metadata": {"source": "lesson.pdf", "page": 4},
}

print(multimodal_record)
# In a real system, embed the text/description for retrieval,
# keep the original image reference, retrieve both, and pass both to the model.

## Evaluation

Use multimodal faithfulness/relevance metrics where applicable, plus retrieval recall for the visual evidence. Check whether the model actually used the relevant image rather than inventing visual details.

## Improvement ideas

Keep image metadata, preserve page/image IDs, use appropriate image detail settings, compress or crop when necessary, and benchmark text-only vs text+image retrieval.

## Practice exercises

1. Create a chart image and ask the model a specific question.
2. Store image descriptions in a vector store.
3. Evaluate whether the correct image is retrieved.

## Multi-modal RAG: what counts as evidence?

A multimodal corpus can contain:
- normal text chunks
- page images
- tables
- charts
- screenshots
- diagrams

The key design choice is whether to retrieve the raw visual artifact, a textual description of it, or both.

## Multi-modal retrieval patterns

**Pattern A: text index + image metadata**

```text
image → description → text embedding
                    ↓
                 retrieve
                    ↓
              original image
```

**Pattern B: separate indexes**

```text
text query → text index
image query → image index
             ↓
           merge
```

**Pattern C: multimodal embedding model**

Text and images are mapped into a shared embedding space. This is powerful but requires a suitable multimodal embedding model and careful evaluation.

## Failure cases

- OCR misses a number in a table.
- A chart is retrieved but the question refers to another chart.
- Page metadata is lost.
- The LLM guesses a visual detail that is not actually present.

Add test cases specifically for these failures.

## Final checklist

- [ ] I can explain the concept in simple words.
- [ ] I can describe where it fits in a GenAI system.
- [ ] I can run and modify the code.
- [ ] I know at least one way to evaluate it.
- [ ] I know what to change when it fails.

## Research / reference notes


- LangChain Python reference: https://reference.langchain.com/python/langchain/overview
- LangGraph Python reference: https://reference.langchain.com/python/langgraph/overview
- Ragas metrics: https://docs.ragas.io/en/latest/concepts/metrics/available_metrics/
- OpenAI function calling guide: https://developers.openai.com/api/docs/guides/function-calling
- OpenAI Responses API: https://developers.openai.com/api/reference/cli/resources/responses/methods/create

- OpenAI image inputs: https://developers.openai.com/api/docs/quickstart
- OpenAI file inputs: https://developers.openai.com/api/docs/guides/file-inputs
- LangChain multimodal message reference: https://reference.langchain.com/python/langchain-core/messages